In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
np.set_printoptions(precision=2, suppress=True)

In [66]:
ratings = pd.read_csv(
    r"C:\Users\user\Downloads\ml-100k\ml-100k\u.data",
    sep="\t",
    names=["user_id" , "movie_id", "rating" , "timestamp"]
)

train,test = train_test_split(ratings, test_size=0.2, random_state=42)

In [67]:
rating_matrix = ratings.pivot(index="user_id", columns="movie_id", values="rating")
all_users = ratings["user_id"].unique()
all_movies = ratings["movie_id"].unique()

In [68]:
train_matrix = train.pivot(index="user_id",columns="movie_id",values="rating").reindex(index=all_users,columns=all_movies)
train_matrix.shape

(943, 1682)

In [69]:
test_matrix = test.pivot(index="user_id",columns="movie_id",values="rating").reindex(index=all_users,columns=all_movies)
test_matrix.shape

(943, 1682)

In [81]:
def split_Y_R(train_matrix,test_matrix):
    
    Y_train = np.array(train_matrix.copy())
    R_train = Y_train.copy()
    
    nan_mask = np.isnan(Y_train)
    
    Y_train[nan_mask] = 0
    R_train[nan_mask] = 0
    R_train[~nan_mask] = 1
    
    Y_test = np.array(test_matrix.copy())
    R_test = Y_test.copy()
    
    nan_mask = np.isnan(Y_test)
    
    Y_test[nan_mask] = 0
    R_test[nan_mask] = 0
    R_test[~nan_mask] = 1

    return Y_train, Y_test, R_train, R_test

In [111]:
film_num = all_movies.shape[0]
user_num = all_users.shape[0]

In [112]:
def costFunc(W, X, B_film, B_user,yArray, rArray):
    error = predict(W, X, B_film, B_user) - yArray
    error = rArray * error
    cost = np.sum(np.square(error)) / 2
    return cost

In [129]:
def predict(W, X, B_film, B_user, global_mean):
    prediction_matrix = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:] + global_mean
    return prediction_matrix

In [114]:
def calculate_Global_Mean(Y,R):
    global_mean = np.sum(Y) / np.sum(R)
    return global_mean

In [115]:
np.array([1,2,5])[np.newaxis,:]

array([[1, 2, 5]])

In [131]:
def gradientDescent(W, X, B_film, B_user, global_mean, yArray, rArray, iteration, learning_rate, momentum = 0.8 ,lambda_ = 0.1):

    
    dW = None; dX = None; dB_film = None; dB_user = None;
    
    vW = np.zeros_like(W)
    vX = np.zeros_like(X)
    vB_film = np.zeros_like(B_film)
    vB_user = np.zeros_like(B_user)
        
    for i in range(iteration + 1):
        
        prediction = predict(W, X, B_film, B_user, global_mean)
        error = (prediction - yArray) * rArray
        
        dW = np.dot(error, X) + lambda_ * W
        dX = np.dot(error.T, W) + lambda_ * X
        
        dB_film = np.sum(error, axis = 0)
        dB_user = np.sum(error, axis = 1)

        vW = momentum * vW + learning_rate * dW
        vX = momentum * vX + learning_rate * dX
        vB_film = momentum * vB_film + learning_rate * dB_film
        vB_user = momentum * vB_user + learning_rate * dB_user
        
        W = W - vW
        X = X - vX
        B_film = B_film - vB_film
        B_user = B_user - vB_user

        if i % 100 == 0:
            data_loss = np.sum( np.square(error) / 2)
            regularization_loss = (lambda_ / 2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
            total_loss = data_loss + regularization_loss
            print(i,"th iteration     Loss:", "{:.4f}".format(total_loss))

    return W, X, B_film, B_user

In [130]:
def calculate_errors(W, X, B_film, B_user, global_mean, yArray, rArray):
    n = np.sum(rArray)
    prediction = predict(W, X, B_film, B_user, global_mean)
    error = (prediction - yArray) * rArray        
    
    mse = np.sum(np.square(error)) / n
    rmse = np.sqrt(mse)
    mae = np.sum(np.abs(error)) / n
    
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("MAE:", mae)

In [122]:
Y_train, Y_test, R_train, R_test = split_Y_R(train_matrix, test_matrix)
global_mean = calculate_Global_Mean(Y_train, R_train)

In [135]:
feature_num = 10
W = np.random.randn(user_num,feature_num) * 0.01
X = np.random.randn(film_num,feature_num) * 0.01
B_film = np.zeros((film_num,))
B_user = np.zeros((user_num,))
W , X, B_film, B_user = gradientDescent(W, X, B_film, B_user, global_mean, Y_train, R_train, 
                                        iteration=500, learning_rate=0.0015, momentum=0.85, lambda_= 0.05)

0 th iteration     Loss: 50725.8889
100 th iteration     Loss: 17752.7656
200 th iteration     Loss: 16564.3468
300 th iteration     Loss: 16243.4884
400 th iteration     Loss: 16085.5391
500 th iteration     Loss: 15989.6959


In [136]:
print("Train")
calculate_errors(W, X, B_film, B_user, global_mean, Y_train, R_train)

Train
MSE: 0.393749981560478
RMSE: 0.6274950052075937
MAE: 0.47910771127623847


In [137]:
print("Test")
calculate_errors(W, X, B_film, B_user, global_mean, Y_test, R_test)

Test
MSE: 1.7055195189966843
RMSE: 1.3059554046737907
MAE: 0.8955223531886913


In [128]:
print(np.linalg.norm(W))
print(np.linalg.norm(X))

83.48450988786611
83.73968481433808
